# FANCY TITLE HERE

**42186 Model-based Machine Learning — final project, group submission.**

This notebook is the single source of truth for our model comparison. It takes the preprocessed Swedish forest dataset (described in the accompanying report) and turns it into a clean modelling support, then trains and compares a sequence of Bayesian models of increasing complexity on the same train/test split.

### What we are predicting

For each $12.5\,\mathrm{m} \times 12.5\,\mathrm{m}$ pixel of forest in a $5\,\mathrm{km} \times 5\,\mathrm{km}$ area of northern Skåne, we predict per-pixel **growth in stem volume between two airborne-lidar inventory cycles** roughly a decade apart:

$$
\Delta V \;=\; \texttt{volume\_t2} \;-\; \texttt{volume\_t1} \quad [\mathrm{m^3/ha}].
$$

The covariates are the cycle-1 forest state (height, diameter, basal area, volume, biomass, vegetation ratio), local hydrology (soil moisture, log-transformed flow accumulation), and a centred-log-ratio (CLR) encoding of the SLU per-species volumes. The PGM and feature rationale are in §3 of the report.

### How this notebook is laid out

| § | What it contains | Why it's here |
|---|---|---|
| 1 | Load the preprocessed CSV, rename Swedish columns to English, drop the identically-zero lodgepole-pine column, and drop non-forest / lake / disturbed pixels via `is_stable_forest`. | Defines the support on which a Gaussian likelihood is well-posed and gives every column a readable name. |
| 2 | Build BK-cell-disjoint train/test splits across a nested scaling grid (`n_cells ∈ {5, 25, 50, all}`). | We need honest evaluation — pixels inside one BK indexruta cell are near-identical, so a random split leaks. The scaling grid lets us read how each model handles "more data of the same kind". |
| 3 | Engineer the 16-column 'standard' feature matrix and the volume-growth target. | Decouples feature choice from any single model. |
| 4 | `slice_step(n_cells)` — one call returns standardised train/test matrices for any scaling step. | Each model below uses the *same* split and the *same* preprocessing, so model differences are not confounded by preprocessing differences. |
| 5 | Evaluation utilities (RMSE / MAE / Bias / R² / correlation + Moran's I on test residuals). | Moran's I detects spatial autocorrelation that the model failed to absorb. This is the diagnostic that justifies adding spatial structure to a model. |
| 6 | Ridge baseline across the scaling grid. | Frequentist sanity check the data flow works end to end. Any Bayesian model that doesn't beat this is suspect. |
| 7 | Bayesian models, smallest to largest: BLR (VI + MCMC), heteroscedastic linear, hierarchical (random intercept), spatial-lag, SAR, BNN, GP variants. | This is the comparison the report's "Models" section discusses. Each one inherits §4–§5 unchanged. |
| 8 | Diagnostics and sanity checks — generative recovery, ELBO traces, inference-method agreement. | **The course brief flags these explicitly.** |

In [18]:
from __future__ import annotations
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

SEED = 42
np.random.seed(SEED)

# Canonical dataset for this notebook: 5 km × 5 km AOI, preprocessed,
# already restricted spatially. See the report's data section for how it
# was derived from the raster pipeline.
CSV_PATH = Path("out_5km_idx_preprocessed.csv")
CSV_URL = (
    "https://github.com/Somon8/mbml-forest-pipeline/"
    "releases/download/v2.0-data-5km/out_5km_idx_preprocessed.csv"
)
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
SPLITS_JSON = CACHE_DIR / "splits_5km.json"

if not CSV_PATH.exists():
    print(f"CSV missing — downloading from {CSV_URL} ...")
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
    print(f"  → wrote {CSV_PATH} ({CSV_PATH.stat().st_size/1e6:.0f} MB)")

## 1. Load and filter

The CSV (`out_5km_idx_preprocessed.csv`) is the preprocessed Swedish-forest dataset at $12.5\,\mathrm{m}$ resolution, restricted to the $5\,\mathrm{km} \times 5\,\mathrm{km}$ AOI we use for modelling. The raster→tabular pipeline that produced it lives outside this notebook and is described in the report's data section.

One filter is applied here:

- **`is_stable_forest`** drops pixels that are non-forest in either inventory cycle, are water (lakes), or lost biomass / height / volume between cycles. This leaves the support on which volume growth ($\Delta V$) is *defined, continuous, and non-negative* — modelling outside this support would force a Gaussian likelihood to absorb a large zero spike for non-forest and a separate negative mass for disturbance, which is exactly the miscalibration the report flags as a structural property of the data.

In [19]:
# Swedish raster column names → readable English. Pipeline outputs and other
# notebooks (04, 06) still use the Swedish names; the rename happens here at
# load time so this notebook is self-contained for a reviewer.
RENAME_MAP = {
    # Skogliga grunddata, cycle 1 ("omdrev 1")
    "biomassa_omdrev1":         "biomass_t1",
    "grundyta_omdrev1":         "basal_area_t1",
    "medeldiameter_omdrev1":    "mean_diameter_t1",
    "medelhojd_omdrev1":        "mean_height_t1",
    "p95_omdrev1":              "p95_height_t1",
    "vegetationskvot_omdrev1":  "veg_ratio_t1",
    "volym_omdrev1":            "volume_t1",
    # Skogliga grunddata, cycle 2 ("omdrev 2")
    "biomassa_omdrev2":         "biomass_t2",
    "grundyta_omdrev2":         "basal_area_t2",
    "medeldiameter_omdrev2":    "mean_diameter_t2",
    "medelhojd_omdrev2":        "mean_height_t2",
    "p95_omdrev2":              "p95_height_t2",
    "vegetationskvot_omdrev2":  "veg_ratio_t2",
    "volym_omdrev2":            "volume_t2",
    # Hydrology, soil and per-pixel canopy
    "flodesackumulering":       "flow_accumulation",
    "markfuktighet":            "soil_moisture",
    "markfuktighet_klassad":    "soil_moisture_class",
    "tradhojd":                 "tree_height",
    # SLU Skogskarta — totals and per-species volumes
    "slu_skogskarta_biomassa":       "slu_total_biomass",
    "slu_skogskarta_grundyta":       "slu_total_basal_area",
    "slu_skogskarta_medeldiameter":  "slu_total_mean_diameter",
    "slu_skogskarta_volym":          "slu_total_volume",
    "slu_skogskarta_gran_volym":     "spruce_volume",
    "slu_skogskarta_tall_volym":     "pine_volume",
    "slu_skogskarta_bjork_volym":    "birch_volume",
    "slu_skogskarta_ek_volym":       "oak_volume",
    "slu_skogskarta_bok_volym":      "beech_volume",
    "slu_skogskarta_ovrigt_volym":   "other_species_volume",
    # `slu_skogskarta_contorta_volym` is identically zero on this AOI and is
    # dropped entirely below — no rename needed.
}

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df_raw = df_raw.rename(columns=RENAME_MAP)
df_raw = df_raw.drop(columns=["slu_skogskarta_contorta_volym"], errors="ignore")
print(f"raw rows           : {len(df_raw):>8,}  ({df_raw['x'].nunique()} × {df_raw['y'].nunique()} grid)")

df = df_raw[df_raw["is_stable_forest"].astype(bool)].reset_index(drop=True)
print(f"stable-forest rows : {len(df):>8,}  ({100*len(df)/len(df_raw):.1f}% of grid)")
print(f"unique BK cells    : {df['BK'].nunique():>8,}")

raw rows           :  160,000  (400 × 400 grid)
stable-forest rows :   63,073  (39.4% of grid)
unique BK cells    :      121


**Reading the output.** The `is_stable_forest` filter removes roughly 60% of pixels — the AOI is a mixed landscape of forest stands, roads, fields, lakes, and recent clear-cuts. The ~63 000 surviving pixels span 121 BK indexruta cells, which become the unit of the train/test split in the next section.

## 2. BK-cell-disjoint splits on a nested scaling grid

**Why splitting by BK indexruta cell.** Skogsstyrelsen publishes a $500\,\mathrm{m} \times 500\,\mathrm{m}$ administrative grid (the *Sverige-indexruta*); every pixel in our CSV is labelled with the BK cell that contains it. Pixels inside one BK cell are near-identical — same forest stand, same management history, same soil class — so a random pixel split would put a held-out pixel next to its ~1 600 quasi-twins on the training side and inflate R² by tens of percent without telling us anything generalisable. **Splitting by BK cell** prevents that within-cell leakage: every test pixel is in a BK cell that contains no training pixel.

**What it does and does not solve.** BK-disjoint splits kill within-cell leakage. They do *not* kill across-cell spatial autocorrelation — two adjacent BK cells are still correlated (similar elevation, similar species mix). The Moran's I statistic in §5 measures the residual autocorrelation a model failed to absorb, and motivates the spatial-component models in §7.

**How train and test are built.** From the 121 BK cells in the AOI:

- **24 cells (≈ 20 %)** are picked once by a seeded random draw and held out as the **test set**. These cells — and every pixel inside them — are never seen by any model during training, at any scaling step.
- **97 cells (the remainder)** form the **training pool**, kept in a seeded random order.

The same 24-cell test set is used everywhere downstream, so test pixels are *constant* across models *and* across scaling steps. RMSEs are therefore directly comparable.

**Why a *nested* scaling grid.** A scaling step `n_cells = k` trains on the **first $k$ cells of the 97-cell training pool**, against the same 24-cell test set. Because the pool is ordered once and never reshuffled, the 5 cells used at `n_cells = 5` are a strict subset of the 25 used at `n_cells = 25`, which are a strict subset of 50, and so on up to `n_cells = 'all'` ($= 97$). The scaling axis therefore measures "more of the same kind of data" rather than "different draw of the same size" — exactly the right experiment to read whether a model is data-starved or capacity-bound.

The grid auto-shrinks to fit the train-pool size: with 97 training cells, `[5, 25, 50]` plus `'all'` are the scaling choices.

In [20]:
TEST_FRAC = 0.20


def build_splits(df: pd.DataFrame, test_frac: float, seed: int) -> dict:
    rng = np.random.default_rng(seed)
    all_bk = sorted(df["BK"].unique().tolist())
    rng.shuffle(all_bk)
    n_test = int(round(test_frac * len(all_bk)))
    return {
        "test_bk": sorted(all_bk[:n_test]),
        "train_bk_ordered": list(all_bk[n_test:]),
    }


def get_train_bk(splits: dict, n_cells) -> list:
    pool = splits["train_bk_ordered"]
    return list(pool) if n_cells == "all" else list(pool[: int(n_cells)])


if SPLITS_JSON.exists():
    splits = json.loads(SPLITS_JSON.read_text())
    print(f"loaded splits from {SPLITS_JSON}")
else:
    splits = build_splits(df, TEST_FRAC, seed=SEED)
    SPLITS_JSON.write_text(json.dumps(splits))
    print(f"built and saved splits → {SPLITS_JSON}")

# Scaling grid auto-shrinks to fit the train pool — every step except 'all' must be
# strictly smaller than the pool so the nested-subset invariant stays meaningful.
n_pool = len(splits["train_bk_ordered"])
SCALING_GRID = [n for n in [5, 25, 50, 100, 250] if n < n_pool] + ["all"]

print(f"test BK cells : {len(splits['test_bk']):>4}")
print(f"train pool BK : {n_pool:>4}")
print(f"scaling grid  : {SCALING_GRID}")
for n in SCALING_GRID:
    tbk = get_train_bk(splits, n)
    n_pix = df[df["BK"].isin(set(tbk))].shape[0]
    print(f"  n_cells={str(n):>4}  → {len(tbk):>4} BK cells, {n_pix:>7,} pixels")

loaded splits from cache\splits_5km.json
test BK cells :   24
train pool BK :   97
scaling grid  : [5, 25, 50, 'all']
  n_cells=   5  →    5 BK cells,   3,411 pixels
  n_cells=  25  →   25 BK cells,  14,945 pixels
  n_cells=  50  →   50 BK cells,  28,925 pixels
  n_cells= all  →   97 BK cells,  52,184 pixels


In [21]:
# nesting + disjointness sanity checks
subsets = [set(get_train_bk(splits, n)) for n in SCALING_GRID]
for i in range(len(subsets) - 1):
    assert subsets[i].issubset(subsets[i + 1]), f"subset {i} not nested in {i+1}"
test_set = set(splits["test_bk"])
for n, ss in zip(SCALING_GRID, subsets):
    assert not (ss & test_set), f"train at n_cells={n} overlaps test BK"
print("nesting + disjointness: OK")

nesting + disjointness: OK


**Reading the output.** 24 BK cells (~20% of the 121 in the window) are held out for testing once and never seen during training. The remaining 97 are the training pool, drawn in increasing chunks for the scaling axis: 5 → 25 → 50 → 97. The pixel counts roughly track the BK count (~540 pixels per cell after the stable-forest filter).

## 3. Feature matrix and target

**Target — volume growth.** $\Delta V = \texttt{volume\_t2} - \texttt{volume\_t1}$ is the per-pixel change in stem volume between the two inventory cycles, in m³/ha. On the stable-forest support it is non-negative (forest stands gain volume between cycles, never lose it — `delta_neg_volym` pixels are excluded), continuous, and right-skewed. A Gaussian likelihood is a defensible *starting point* — the report and §7 will discuss whether richer likelihoods (heteroscedastic, log-normal, Gamma) calibrate the tail better.

**Features — 16 columns, three logical blocks:**

1. **Forest state at cycle 1 (10 cols, raw):** `biomass_t1`, `basal_area_t1`, `mean_diameter_t1`, `mean_height_t1`, `p95_height_t1`, `veg_ratio_t1`, `volume_t1` (the seven lidar-derived cycle-1 forest summaries), `flow_accumulation` (upstream drainage), `soil_moisture` (SLU soil-moisture index), and `slu_total_biomass` (total above-ground biomass from the SLU Skogskarta product, t/ha). These are the *initial conditions* the model uses to predict growth between cycles. Cycle-2 columns are deliberately **excluded** — they would leak the target.
2. **Species composition (6 cols, CLR-transformed):** the six SLU per-species stem volumes that actually occur on this AOI — `spruce_volume`, `pine_volume`, `birch_volume`, `oak_volume`, `beech_volume`, `other_species_volume`. They sum to the SLU total volume by construction, so they live on the 6-simplex rather than in $\mathbb{R}^6$. The centred log-ratio transform $p_i \mapsto \log(p_i / g(p))$, where $g(p)$ is the row's geometric mean of the species proportions, lifts the block into an unconstrained space where standard linear-model machinery applies cleanly. A small $\varepsilon$ keeps the log finite when a species is locally absent. Lodgepole pine (`slu_skogskarta_contorta_volym`) is identically zero across this AOI and is dropped at load time rather than included via an $\varepsilon$ fallback — it would carry no signal and would distort the geometric mean.

**A subtle leakage point.** `volume_t1` is in the feature block *and* is one of the two ingredients of the target. That's intentional and not leakage: we are explicitly modelling growth conditional on the initial volume, exactly as a textbook growth equation $\Delta V = f(V_1, \text{covariates}) - V_1$ would. The cycle-1 volume is observable a decade before the cycle-2 measurement, so a forecasting model legitimately has it.

**Standardisation happens later.** Both $\mathbf{X}$ and $y$ are mean-centred and unit-scaled inside `slice_step` using **training-set statistics only**, so test pixels never contribute to the standardisation scale. The unstandardised arrays are kept here as the source of truth.

In [22]:
BASE_COLS = [
    "biomass_t1", "basal_area_t1", "mean_diameter_t1",
    "mean_height_t1", "p95_height_t1", "veg_ratio_t1",
    "volume_t1", "flow_accumulation", "soil_moisture",
    "slu_total_biomass",
]
SPECIES_COLS = [
    "spruce_volume", "pine_volume", "birch_volume",
    "oak_volume", "beech_volume", "other_species_volume",
]


def clr_transform(values: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    row_sums = values.sum(axis=1, keepdims=True)
    nz = row_sums.ravel() > 0
    props = np.where(row_sums > 0, values / np.where(row_sums > 0, row_sums, 1.0), 0.0)
    props_safe = np.where(props <= 0, eps, props)
    log_props = np.log(props_safe)
    clr = log_props - log_props.mean(axis=1, keepdims=True)
    clr[~nz] = 0.0
    return clr


X_base = df[BASE_COLS].to_numpy(float)
X_clr = clr_transform(df[SPECIES_COLS].to_numpy(float))
X_all = np.hstack([X_base, X_clr])
FEATURE_NAMES = BASE_COLS + [c + "_clr" for c in SPECIES_COLS]

# Target: per-pixel change in stem volume between inventory cycles (m³/ha).
# is_stable_forest excludes delta_neg_volym, so the target is ≥ 0 on the support.
y_all = (df["volume_t2"] - df["volume_t1"]).to_numpy(float)
coords_all = df[["x", "y"]].to_numpy(float)
bk_all = df["BK"].to_numpy()

print(f"X_all   : {X_all.shape}  ({len(FEATURE_NAMES)} features = {len(BASE_COLS)} base + {len(SPECIES_COLS)} CLR)")
print(f"y_all   : {y_all.shape}  mean={y_all.mean():.2f} m³/ha  std={y_all.std():.2f} m³/ha")
print(f"coords  : {coords_all.shape}")

X_all   : (63073, 16)  (16 features = 10 base + 6 CLR)
y_all   : (63073,)  mean=52.07 m³/ha  std=44.57 m³/ha
coords  : (63073, 2)


**Reading the output.** Design matrix is $\sim\!63\,000 \times 17$. 17 features per pixel, three feature blocks combined as described above. The target's mean ($\sim\!52\,\mathrm{m^3/ha}$) and standard deviation ($\sim\!45\,\mathrm{m^3/ha}$) are both physically plausible for a decade of stem-volume accumulation in Swedish managed forest — typical mean-annual-increment figures of $5{-}10\,\mathrm{m^3/ha/year}$ over $\sim\!10$ years bracket the observed mean. The numbers being signed positive on the stable-forest support also confirms that cycle 2 is genuinely the later inventory cycle.

## 4. `slice_step` — one call returns one scaling step's data

Every model in §6–§7 reads its training and test data through this single helper, to ensure comparability across models.

`slice_step(n_cells)` returns a dict containing:

- `X_train`, `y_train`, `X_test`, `y_test` — **standardised** against training-set mean and standard deviation only;
- `coords_test` — the (x, y) test coordinates, needed for the Moran's I residual diagnostic;
- `y_mean`, `y_std` — the rescaling constants, so a model that predicts in standardised units can be brought back to decimetres for human-readable error metrics.

In [23]:
def slice_step(n_cells):
    train_bk = set(get_train_bk(splits, n_cells))
    test_bk = set(splits["test_bk"])

    tr = np.isin(bk_all, list(train_bk))
    te = np.isin(bk_all, list(test_bk))

    Xtr, ytr = X_all[tr], y_all[tr]
    Xte, yte = X_all[te], y_all[te]
    coords_te = coords_all[te]

    x_mean = Xtr.mean(axis=0)
    x_std = Xtr.std(axis=0) + 1e-8
    y_mean, y_std = float(ytr.mean()), float(ytr.std() + 1e-8)

    Xtr_s = (Xtr - x_mean) / x_std
    Xte_s = (Xte - x_mean) / x_std
    ytr_s = (ytr - y_mean) / y_std
    yte_s = (yte - y_mean) / y_std

    return {
        "X_train": Xtr_s, "y_train": ytr_s,
        "X_test":  Xte_s, "y_test":  yte_s,
        "coords_test": coords_te,
        "y_mean": y_mean, "y_std": y_std,
        "n_train": int(tr.sum()), "n_test": int(te.sum()),
    }

## 5. Evaluation utilities

All metrics are computed in the original units (m³/ha) after un-standardising the predictions, so the numbers are directly comparable to the target's natural scale.

**Point-prediction metrics.** RMSE, MAE, Bias, $R^2$, and correlation are the standard regression scorecard. We report them all because they answer slightly different questions: RMSE penalises large errors quadratically (relevant if growth outliers matter), MAE is robust and unitful (easy to interpret as "typical error in m³/ha"), Bias reveals systematic over/under-prediction, $R^2$ summarises variance explained, and correlation is scale-invariant.

**Moran's I on test residuals.** The diagnostic that justifies most of the spatial structure in §7. It measures whether the residuals $r_i = \hat{y}_i - y_i$ are spatially autocorrelated — whether neighbouring test pixels have similar errors. We use 8 nearest neighbours in projected $(x, y)$ space with row-standardised weights, so the formula simplifies to

$$ I \;=\; \frac{\sum_i z_i \cdot \overline{z_{\,N(i)}}}{\sum_i z_i^2}, \qquad z_i = r_i - \bar r, $$

where $\overline{z_{\,N(i)}}$ is the mean residual over $i$'s 8 nearest test neighbours. The $p$-value comes from a 999-step permutation test: shuffle the residuals on top of the same coordinates many times, recompute $I$, and count how often it exceeds the observed value. The implementation is ~25 lines of `numpy` + `sklearn.NearestNeighbors` so the notebook has no `libpysal` / `esda` dependency.

- $I \approx 0$ with non-significant $p$ ⟹ the model has absorbed the spatial structure; residuals look like noise.
- $I$ substantially positive ⟹ the model is missing spatial information; *some* feature you didn't include or *some* dependency structure you didn't model is varying smoothly across space.

For a covariate-only Bayesian linear model (no spatial component, no hierarchy) we expect $I$ to be **substantially positive** on this data — that's the gap the spatial models in §7 try to close, and is the in-brief motivation (Bayesian Spatial Count Models example) for the spatial-error / correlated-noise term $\phi_i$.

In [24]:
from sklearn.neighbors import NearestNeighbors


def moran_i(values: np.ndarray, coords: np.ndarray, k: int = 8,
            n_permutations: int = 999, seed: int = 42) -> tuple[float, float]:
    """Moran's I on k-NN, row-standardised weights; permutation p-value.

    With row-standardised weights the formula reduces to
        I = sum_i z_i * mean(z_j for j in neighbours_i) / sum_i z_i^2,
    so we never materialise the weight matrix.
    """
    coords = np.asarray(coords)
    values = np.asarray(values).ravel()
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    neighbours = idx[:, 1:]  # drop self

    def _i(z: np.ndarray) -> float:
        num = (z * z[neighbours].mean(axis=1)).sum()
        return float(num / (z @ z))

    z_obs = values - values.mean()
    observed = _i(z_obs)

    rng = np.random.default_rng(seed)
    perms = np.fromiter(
        (_i(rng.permutation(z_obs)) for _ in range(n_permutations)),
        dtype=float, count=n_permutations,
    )
    # One-sided permutation p-value (positive autocorrelation is the alternative
    # we expect; libpysal calls this p_sim with this same convention).
    p = (1 + (perms >= observed).sum()) / (1 + n_permutations)
    return observed, float(p)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, coords: np.ndarray | None = None) -> dict:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    res = y_pred - y_true
    out = {
        "RMSE": float(np.sqrt(np.mean(res ** 2))),
        "MAE":  float(np.mean(np.abs(res))),
        "Bias": float(res.mean()),
        "R2":   float(r2_score(y_true, y_pred)),
        "Corr": float(np.corrcoef(y_true, y_pred)[0, 1]),
    }
    if coords is not None:
        I, p = moran_i(res, coords, k=8)
        out["MoranI"] = I
        out["MoranP"] = p
    return out

## 6. Sanity-check baseline — Ridge across the scaling grid

A frequentist baseline establishes (a) that the data pipeline runs end-to-end, (b) the design matrix is conditioned well enough to fit, and (c) what RMSE a non-Bayesian, non-spatial, regularised linear regression can hit on this problem. Bayesian regularisation should be at least as good as Ridge's $\ell_2$ shrinkage.

**What to look for in the output table.**

- **RMSE should be roughly stable** across the scaling grid. With $\sim\!52\,\mathrm{m^3/ha}$ mean and $\sim\!45\,\mathrm{m^3/ha}$ std on the target, an RMSE in the low-to-mid 30s indicates the linear model captures roughly two-thirds of the variance. A linear model is undercapacity relative to the data size; further drops from richer structure are where the §7 Bayesian extensions should earn their keep.
- **Bias should be small** (a few percent of $y$'s standard deviation). A systematic bias would signal a target-scale or standardisation bug.
- **Moran's I will be positive** at every scaling step — Ridge has no mechanism to absorb spatial autocorrelation. That residual signal is exactly what the §7 spatial models are designed to pick up.

In [25]:
rows = []
for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    model = Ridge().fit(s["X_train"], s["y_train"])
    yhat_s = model.predict(s["X_test"])

    # back to m³/ha for interpretable units
    yhat = yhat_s * s["y_std"] + s["y_mean"]
    ytrue = s["y_test"] * s["y_std"] + s["y_mean"]

    m = evaluate(ytrue, yhat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], **m}
    rows.append(m)

pd.DataFrame(rows)

,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,33.932384,24.523554,-0.541249,0.376298,0.614323,0.181195,0.001
1,25,14945,10889,34.080883,24.600274,1.690629,0.370827,0.611481,0.181784,0.001
2,50,28925,10889,33.732516,24.510800,1.825984,0.383624,0.621459,0.181263,0.001
3,all,52184,10889,33.718425,24.520340,1.974770,0.384139,0.621680,0.178946,0.001


**Reading the output.** RMSE sits in a narrow band of $\sim\!34.2$–$34.6\,\mathrm{m^3/ha}$ across the scaling grid — essentially flat from `n_cells=5` to `n_cells='all'`. The linear model has reached its capacity ceiling on this feature set; more data is not going to help any *linear* model. The improvement for our models therefore lies in richer structure (nonlinear, hierarchical, spatial) rather than more rows. Moran's I sits at $\sim\!0.20$ with $p=0.001$ at every scaling step — strong, statistically significant residual spatial autocorrelation that is invariant under scaling, confirming Ridge cannot model spatial structure and that the §7 spatial models have something real to absorb. Bias is small (under $\sim\!1.5\,\mathrm{m^3/ha}$ at every step, well under 4% of $y$'s std), so the standardisation and rescaling round-trip is correct.

## 7. Bayesian models — building up from simple to complex

This is the comparison the report's *Models* section discusses. Models are introduced in deliberate order, each one motivated by a specific shortcoming of the previous one. Every model uses `slice_step(n_cells)` for its data and `evaluate(...)` for its scoring, so differences between models are not confounded by differences in preprocessing.

### Story arc

| § | Model | What it adds over the previous | What it tests |
|---|---|---|---|
| 7.1 | **BLR via VI** (Pyro `AutoMultivariateNormal`) | Replaces Ridge's point estimate with a full posterior; turns regularisation into priors. | Does VI converge on this likelihood / prior? ELBO trace should be monotone-ish. |
| 7.2 | **BLR via MCMC** (Pyro NUTS) | Asymptotically exact posterior instead of a Gaussian VI approximation. | Inference-method agreement: VI and MCMC should give similar posterior means and intervals on the same model. Disagreement ⟹ VI's mean-field assumption is biting. Required by the course brief. |
| 7.3 | **Heteroscedastic linear** | Lets the observation variance depend on $\mathbf{X}$, $\sigma^2(\mathbf{X})$, instead of being constant. | Does growth's variance increase with tree size (it should — taller trees have bigger annual variation)? |
| 7.4 | **Hierarchical (random intercept per BK)** | Partial pooling per indexruta cell — admits that some 500 m blocks just grow faster than others for reasons outside our covariates. | Does the random-intercept variance dominate the residual variance? If so, the per-BK story is doing real work. |
| 7.5 | **Spatial-lag covariates** | Augments each row with the mean of its rook neighbours' features. | Does explicit local context help over a per-cell intercept? |
| 7.6 | **SAR ($y = \rho W y + X \beta + \varepsilon$)** | Spatial autoregression of the response on neighbour predictions. The Bayesian Spatial Count Models example's correlated noise term, applied to the mean function rather than the noise. | Should drop Moran's I substantially if it's working. |
| 7.7 | **BNN with mean-field SVI** | Nonlinear $f(\mathbf{X})$ instead of $\beta^T \mathbf{X}$. Bayesian neural net, two hidden layers. | Does nonlinearity matter for this feature set? Or does the relationship really look linear? |
| 7.8 | **GP variants** (exact GP at small `n_cells`, SVGP at full) | Nonparametric prior over functions with built-in length scales for spatial smoothness. | Direct Bayesian alternative to SAR for spatial structure; ARD lengthscales tell us which features matter. |

Each subsection follows the same template:

1. *Why this model* — one-paragraph motivation referencing the previous model's shortcoming or a course concept.
2. *Model definition* — Pyro `model()` and `guide()` (or MCMC kernel) with priors named and briefly justified.
3. *Fit + diagnostics* — ELBO trace for VI, $\hat{R}$ + effective sample size for MCMC.
4. *Predict + evaluate* — through `slice_step` / `evaluate`, results appended to a comparison table.
5. *Reading the result* — one short markdown cell saying what we learned and what the next model attempts.

Models land here as separate sub-sections rather than all in one cell so a reviewer can follow the comparison without having to scroll between definitions and results.

### Model 1. Exact Bayesian Inference



### Exact Bayesian inference as a reference model

This cell implements the conjugate Normal-Inverse-Gamma Bayesian linear regression used as the analytically tractable benchmark in this notebook. It takes one `slice_step(n_cells)` result, fits the posterior in closed form, draws posterior samples for the intercept and coefficients, and evaluates the held-out predictions with `evaluate(...)` on the original scale.

Because the model has no spatial term, its test Moran's I is a useful baseline for the later spatial and hierarchical models: if residual autocorrelation stays high here, the more structured models should have something real to absorb.


In [26]:
def exact_bayesian_linear_regression(step_data, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2):
    """Closed-form Bayesian linear regression for one `slice_step(...)` result.

    Parameters
    ----------
    step_data : dict
        Output from `slice_step(n_cells)`. Expected keys are:
        `X_train`, `y_train`, `X_test`, `y_test`, `coords_test`, `y_mean`, `y_std`.
    n_samples : int
        Number of posterior draws to use for the posterior predictive mean.
    tau : float
        Prior scale for the regression coefficients.
    a0, b0 : float
        Hyperparameters of the inverse-gamma prior on the noise variance.

    Returns
    -------
    dict
        Posterior samples, posterior predictive mean on the original scale, and
        the evaluation metrics computed with the notebook's `evaluate(...)` helper.
    """
    X_train = np.asarray(step_data["X_train"])
    y_train = np.asarray(step_data["y_train"]).ravel()
    X_test = np.asarray(step_data["X_test"])
    y_test = np.asarray(step_data["y_test"]).ravel()
    coords_test = np.asarray(step_data["coords_test"])
    y_mean = float(step_data["y_mean"])
    y_std = float(step_data["y_std"])

    n_obs, n_features = X_train.shape
    X_train_design = np.hstack([np.ones((n_obs, 1)), X_train])
    X_test_design = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

    n_params = n_features + 1
    mu0 = np.zeros(n_params)
    V0 = (tau ** 2) * np.eye(n_params)
    V0_inv = np.linalg.inv(V0)

    XtX = X_train_design.T @ X_train_design
    Xty = X_train_design.T @ y_train

    Vn_inv = XtX + V0_inv
    Vn = np.linalg.inv(Vn_inv)
    mu_n = Vn @ Xty

    a_n = a0 + n_obs / 2.0
    quadratic_term = (y_train.T @ y_train) + (mu0.T @ V0_inv @ mu0) - (mu_n.T @ Vn_inv @ mu_n)
    b_n = b0 + 0.5 * quadratic_term

    gamma_draws = np.random.gamma(shape=a_n, scale=1.0 / b_n, size=n_samples)
    sigma2_samples = 1.0 / gamma_draws

    chol_Vn = np.linalg.cholesky(Vn)
    theta_samples = np.empty((n_samples, n_params))
    for i, sigma2 in enumerate(sigma2_samples):
        latent_draw = np.random.normal(size=n_params)
        theta_samples[i] = mu_n + np.sqrt(sigma2) * (chol_Vn @ latent_draw)

    y_hat_std = (X_test_design @ theta_samples.T).mean(axis=1)
    y_hat = y_hat_std * y_std + y_mean
    y_true = y_test * y_std + y_mean
    metrics = evaluate(y_true, y_hat, coords=coords_test)

    return {
        "theta_samples": theta_samples,
        "sigma2_samples": sigma2_samples,
        "y_hat_std": y_hat_std,
        "y_hat": y_hat,
        "y_true": y_true,
        "metrics": metrics,
        "posterior_mean_theta": mu_n,
        "posterior_cov_theta": Vn,
    }


# Run exact inference across the scaling grid, collect results like the Ridge baseline.
rows = []
for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    exact_result = exact_bayesian_linear_regression(s, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2)
    
    m = exact_result["metrics"]
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], **m}
    rows.append(m)

pd.DataFrame(rows)

,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,33.921905,24.516541,-0.495750,0.376683,0.614876,0.178394,0.001
1,25,14945,10889,34.073374,24.590349,1.678498,0.371105,0.611753,0.180892,0.001
2,50,28925,10889,33.731277,24.508398,1.834485,0.383669,0.621540,0.180873,0.001
3,all,52184,10889,33.716478,24.518734,1.975397,0.384210,0.621744,0.178661,0.001


### Model 2. Bayesian Linear Regression via Stochastic Variational Inference

**Why SVI over exact inference.** The exact conjugate posterior in Model 1 is analytically tractable but requires a closed-form posterior—a luxury not available for hierarchical, spatial, or nonlinear models. Stochastic Variational Inference (SVI) turns Bayesian inference into an optimization problem: we specify a parametric variational family $q(\theta \mid \phi)$ (the *guide*) and find the parameters $\phi$ that minimise the KL divergence $\mathrm{KL}(q \mid\mid p)$, equivalently maximise the Evidence Lower Bound (ELBO). Unlike exact inference, SVI scales to arbitrarily complex models and large datasets (via minibatching) at the cost of a biased (but consistent) posterior approximation.

**Implementation.** This cell uses Pyro's `AutoMultivariateNormal` guide, which automatically constructs a mean-field Gaussian variational posterior over all model parameters. We use `ClippedAdam` optimizer with the standard Trace_ELBO loss, stepping over 1000 iterations per scaling step. Results are collected in the same format as the Ridge and exact-inference baselines so all three are directly comparable. If VI's mean-field assumption is tight, the posterior means and credible intervals should agree closely with the exact inference results.
 

**Why SVI over exact inference.** The exact conjugate posterior in Model 1 is analytically tractable but requires a closed-form posterior—a luxury not available for hierarchical, spatial, or nonlinear models. Stochastic Variational Inference (SVI) turns Bayesian inference into an optimization problem: we specify a parametric variational family $q(\theta \mid \phi)$ (the *guide*) and find the parameters $\phi$ that minimise the KL divergence $\mathrm{KL}(q \mid\mid p)$, equivalently maximise the Evidence Lower Bound (ELBO). Unlike exact inference, SVI scales to arbitrarily complex models and large datasets (via minibatching) at the cost of a biased (but consistent) posterior approximation.

In [27]:
# Import Pyro and torch for SVI
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


def blr_model_pyro(X, y=None):
    """Bayesian linear regression model in Pyro.
    
    Prior on intercept: Normal(0, 1)
    Prior on coefficients: Normal(0, 1)  — standardised to match normalized data
    Prior on noise scale: HalfNormal(1) — scaled to standardized target units
    """
    n, d = X.shape
    alpha = pyro.sample("alpha", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean = alpha + X @ beta
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run SVI across the scaling grid, collect results like Ridge and exact inference.
rows_svi = []

for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup SVI
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    guide = AutoDiagonalNormal(blr_model_pyro)
    optimizer = Adam({"lr": 0.01})
    svi = SVI(blr_model_pyro, guide, optimizer, loss=Trace_ELBO())
    
    # Training loop: optimize ELBO
    n_steps = 2000
    elbos = []
    for step in range(n_steps):
        elbo = svi.step(X_train_torch, y_train_torch)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] completed {n_steps} steps; final ELBO = {elbos[-1]:10.2f}     ")
    
    # Posterior predictions: sample parameters from guide, compute mean predictions
    with torch.no_grad():
        n_samples = 1000
        y_pred_samples = []
        for _ in range(n_samples):
            # Sample parameters from the guide (posterior approximation)
            guide_trace = pyro.poutine.trace(guide).get_trace(X_train_torch, y_train_torch)
            alpha_sample = guide_trace.nodes["alpha"]["value"]
            beta_sample = guide_trace.nodes["beta"]["value"]
            # Compute mean prediction (without noise) on test set
            y_pred = alpha_sample + X_test_torch @ beta_sample
            y_pred_samples.append(y_pred)
        
        # Average predictions across posterior samples
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "final_elbo": float(elbos[-1]), **m}
    rows_svi.append(m)

print("\nSVI Results across scaling grid:")
pd.DataFrame(rows_svi)

  [5] completed 2000 steps; final ELBO =    3940.79     
  [25] completed 2000 steps; final ELBO =   17863.88     
  [50] completed 2000 steps; final ELBO =   34323.60     
  [all] completed 2000 steps; final ELBO =   61460.48     

SVI Results across scaling grid:


,n_cells,n_train,n_test,final_elbo,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,3940.787266,34.411595,24.646367,-0.636305,0.358557,0.599056,0.195506,0.001
1,25,14945,10889,17863.883252,34.325040,24.664921,1.140254,0.361780,0.602069,0.195427,0.001
2,50,28925,10889,34323.604637,34.220356,24.622264,1.435277,0.365667,0.605626,0.195361,0.001
3,all,52184,10889,61460.481305,34.229703,24.710449,1.685507,0.365321,0.605697,0.193364,0.001


**Reading the output.** The SVI results should be comparable to the exact conjugate inference (Model 1) if the mean-field Gaussian variational family is a good approximation to the posterior. Systematic disagreements in RMSE, MAE, or R² between SVI and exact inference indicate that `AutoDiagonalNormal`'s independence assumption is too restrictive. The ELBO values increase with training set size (more data, larger ELBO), which is expected. Moran's I on residuals should again show significant spatial autocorrelation (~0.18), motivating the hierarchical and spatial models that follow.

### Model 3. Bayesian Linear Regression via MCMC (No-U-Turn Sampler)

**Why MCMC after SVI.** SVI's mean-field guide $q(\theta \mid \phi)$ is a fast but potentially biased approximation to the true posterior. MCMC, specifically the No-U-Turn Sampler (NUTS), is a gold-standard sampling method that (in the limit of infinite computation) draws asymptotically exact samples from $p(\theta \mid y, X)$ without requiring a parametric approximation. Running MCMC on the *same model* after SVI serves two purposes: (i) **inference-method agreement**—checking whether posterior means and credible intervals from MCMC and SVI agree, diagnosing whether VI's independence assumption is too restrictive; (ii) **posterior samples for diagnostics**—MCMC gives correlated samples suitable for uncertainty quantification and posterior predictive checks.

**Implementation.** This cell uses Pyro's `NUTS` kernel (a variant of Hamiltonian Monte Carlo) with 500 warmup steps and 500 post-warmup samples per scaling step. To keep computation tractable across the full scaling grid, we run MCMC for smaller `n_cells` values; at `n_cells='all'` we use fewer samples. Results are collected in the same format as SVI and exact inference for direct comparison.

In [28]:
# Import MCMC and NUTS if not already imported
from pyro.infer import MCMC, NUTS

# ===== USER SETTING: Set to False to run only on smallest scaling step (faster for testing) =====
RUN_MCMC_FULL_GRID = False  # Set to True to run full scaling grid
# ============================================================================================

# Reuse the same model function as SVI (defined earlier, but reproduced here for clarity)
def blr_model_mcmc(X, y=None):
    """Bayesian linear regression model for MCMC.
    
    Same priors as SVI Model 2 for direct comparability.
    """
    n, d = X.shape
    alpha = pyro.sample("alpha", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    sigma = pyro.sample("sigma", dist.HalfNormal(torch.tensor(1.0)))
    mean = alpha + X @ beta
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run MCMC across the scaling grid, collect results like Ridge, exact inference, and SVI.
rows_mcmc = []
scaling_steps = SCALING_GRID if RUN_MCMC_FULL_GRID else [SCALING_GRID[0]]

print(f"Running MCMC on: {scaling_steps}")
print(f"(Set RUN_MCMC_FULL_GRID = True to run full scaling grid)\n")

for n_cells in scaling_steps:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup MCMC
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    # Adapt sampler parameters based on training set size (more data → fewer warmup steps needed)
    if n_cells == 5:
        warmup_steps = 500
        num_samples = 500
    elif n_cells == 25:
        warmup_steps = 400
        num_samples = 400
    elif n_cells == 50:
        warmup_steps = 300
        num_samples = 300
    else:  # n_cells == 'all'
        warmup_steps = 200
        num_samples = 200
    
    kernel = NUTS(blr_model_mcmc)
    mcmc = MCMC(kernel, num_samples=num_samples, warmup_steps=warmup_steps, num_chains=1)
    
    print(f"Running MCMC for n_cells={n_cells}: {warmup_steps} warmup, {num_samples} samples")
    mcmc.run(X_train_torch, y_train_torch)
    
    # Extract posterior samples
    posterior_samples = mcmc.get_samples()
    
    # Posterior predictions: average predictions across posterior samples (mean predictions without noise)
    with torch.no_grad():
        alpha_samples = posterior_samples["alpha"]  # shape: (num_samples,)
        beta_samples = posterior_samples["beta"]    # shape: (num_samples, n_features)
        
        # Compute predictions for each posterior sample and average
        y_pred_samples = []
        for i in range(alpha_samples.shape[0]):
            alpha_i = alpha_samples[i]
            beta_i = beta_samples[i]
            y_pred_i = alpha_i + X_test_torch @ beta_i
            y_pred_samples.append(y_pred_i)
        
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "n_mcmc_samples": int(num_samples), **m}
    rows_mcmc.append(m)
    print(f"  → RMSE: {m['RMSE']:.2f}, R2: {m['R2']:.4f}\n")

print("\nMCMC Results across scaling grid:")
pd.DataFrame(rows_mcmc)

Running MCMC on: [5]
(Set RUN_MCMC_FULL_GRID = True to run full scaling grid)

Running MCMC for n_cells=5: 500 warmup, 500 samples


Sample: 100%|██████████| 1000/1000 [13:18,  1.25it/s, step size=1.79e-02, acc. prob=0.914]


  → RMSE: 33.92, R2: 0.3766


MCMC Results across scaling grid:


,n_cells,n_train,n_test,n_mcmc_samples,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,500,33.924225,24.522583,-0.510713,0.376598,0.614653,0.17999,0.001


**Reading the output.** Compare MCMC results to SVI results in Model 2. Posterior means (via averaged predictions) should be very similar if the mean-field assumption in SVI is tight; large differences suggest SVI's independence assumption is biting. RMSE and R² should be comparable across both methods. The `n_mcmc_samples` column tracks how many post-warmup samples were drawn (adaptive to dataset size to keep runtime reasonable). MCMC is slower than SVI but provides exact posterior samples—if inference-method agreement is good, SVI is validated as a fast approximation for downstream models (hierarchical, spatial, nonlinear).

### Model 4. Heteroscedastic Bayesian Linear Regression

**Why heteroscedastic variance.** Models 1–3 assume a constant observation variance $\sigma^2$ across all training data — implicitly treating all pixels' growth measurements as equally precise. This is restrictive: in reality, growth variance often depends on the forest state. For example, larger trees have larger absolute growth increments and their year-to-year variation is consequently larger. A heteroscedastic model lets the noise variance be a function of the covariates, $\sigma^2(\mathbf{x})$, fitting a second linear predictor to $\log(\sigma^2)$ and learning which features drive measurement or process noise. If heteroscedasticity is present and unmodeled, residual diagnostics should reveal non-constant variance across covariate space; capturing it can improve calibration and uncertainty quantification.

**Implementation.** This cell uses Pyro with a heteroscedastic likelihood: both the mean $\mu(\mathbf{x}) = \alpha_\mu + \mathbf{x}^T \boldsymbol{\beta}_\mu$ and the log-variance $\log(\sigma^2(\mathbf{x})) = \alpha_v + \mathbf{x}^T \boldsymbol{\beta}_v$ are linear functions of $\mathbf{x}$. The variance is parameterized via the log to ensure positivity. We use `AutoDiagonalNormal` as the guide to keep the optimization well-conditioned (like Model 2), and SVI with 2000 steps per scaling step. Predictions average over 1000 posterior samples, and the same `evaluate(...)` function scores the held-out test set.

In [29]:
def heteroscedastic_model_pyro(X, y=None):
    """Heteroscedastic Bayesian linear regression in Pyro.
    
    Both mean and variance are linear functions of X.
    - Mean: mu(X) = alpha_mu + X @ beta_mu
    - Log-variance: log_sigma2(X) = alpha_v + X @ beta_v
    
    All priors are Normal(0, 1) scaled to standardized data.
    """
    n, d = X.shape
    
    # Priors for mean function
    alpha_mu = pyro.sample("alpha_mu", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta_mu = pyro.sample(
        "beta_mu",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    
    # Priors for log-variance function (allowing noise to depend on X)
    alpha_v = pyro.sample("alpha_v", dist.Normal(torch.tensor(0.0), torch.tensor(1.0)))
    beta_v = pyro.sample(
        "beta_v",
        dist.Normal(torch.zeros(d), torch.ones(d)).to_event(1)
    )
    
    # Heteroscedastic likelihood: variance depends on X
    mean = alpha_mu + X @ beta_mu
    log_sigma2 = alpha_v + X @ beta_v
    sigma = torch.exp(0.5 * log_sigma2)  # convert log-variance to std deviation
    
    with pyro.plate("data", n):
        return pyro.sample("y", dist.Normal(mean, sigma), obs=y)


# Run heteroscedastic SVI across the scaling grid
rows_hetero = []

for n_cells in SCALING_GRID:
    s = slice_step(n_cells)
    
    # Convert to torch tensors
    X_train_torch = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train_torch = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test_torch = torch.tensor(s["X_test"], dtype=torch.float32)
    
    # Setup SVI
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)
    
    guide = AutoDiagonalNormal(heteroscedastic_model_pyro)
    optimizer = Adam({"lr": 0.01})
    svi = SVI(heteroscedastic_model_pyro, guide, optimizer, loss=Trace_ELBO())
    
    # Training loop: optimize ELBO
    n_steps = 2000
    elbos = []
    for step in range(n_steps):
        elbo = svi.step(X_train_torch, y_train_torch)
        elbos.append(elbo)
        if step % 400 == 0:
            print(f"  [{n_cells}] step {step:4d}: ELBO = {elbo:10.2f}", end="\r")
    print(f"  [{n_cells}] completed {n_steps} steps; final ELBO = {elbos[-1]:10.2f}     ")
    
    # Posterior predictions: sample parameters from guide, compute mean predictions
    with torch.no_grad():
        n_samples = 1000
        y_pred_samples = []
        for _ in range(n_samples):
            # Sample parameters from the guide (posterior approximation)
            guide_trace = pyro.poutine.trace(guide).get_trace(X_train_torch, y_train_torch)
            alpha_mu_sample = guide_trace.nodes["alpha_mu"]["value"]
            beta_mu_sample = guide_trace.nodes["beta_mu"]["value"]
            # Compute mean prediction (using posterior mean function, ignoring variance samples)
            y_pred = alpha_mu_sample + X_test_torch @ beta_mu_sample
            y_pred_samples.append(y_pred)
        
        # Average predictions across posterior samples
        y_hat_std = torch.stack(y_pred_samples).mean(dim=0).numpy()
    
    # Back to original units (m³/ha)
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]
    
    # Evaluate
    m = evaluate(y_true, y_hat, coords=s["coords_test"])
    m = {"n_cells": n_cells, "n_train": s["n_train"], "n_test": s["n_test"], 
         "final_elbo": float(elbos[-1]), **m}
    rows_hetero.append(m)

print("\nHeteroscedastic Results across scaling grid:")
pd.DataFrame(rows_hetero)

  [5] completed 2000 steps; final ELBO =    3321.67     
  [25] completed 2000 steps; final ELBO =   16495.60     
  [50] completed 2000 steps; final ELBO =   31854.03     
  [all] completed 2000 steps; final ELBO =   56687.87     

Heteroscedastic Results across scaling grid:


,n_cells,n_train,n_test,final_elbo,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,5,3411,10889,3321.667500,36.351223,24.895316,-1.443974,0.284209,0.535592,0.214754,0.001
1,25,14945,10889,16495.601256,36.452982,25.193791,0.485325,0.280196,0.535452,0.220913,0.001
2,50,28925,10889,31854.034693,35.341236,24.857022,0.413574,0.323432,0.568905,0.212179,0.001
3,all,52184,10889,56687.870745,35.330005,24.906366,0.820512,0.323862,0.569487,0.209355,0.001


**Reading the output.** Compare the heteroscedastic RMSE, MAE, and R² to Models 1–3 (which use constant variance). If heteroscedasticity is real in the data, the heteroscedastic model should: (i) have a competitive or better RMSE (if it's just fitting noise, RMSE stays the same or degrades); (ii) produce a lower final ELBO than homoscedastic models on the same training set (more parameters, more flexibility to explain data); (iii) show residuals with different spread at different feature values — check this visually by plotting $|\hat{r}_i|$ vs. predicted $\hat{y}_i$ or vs. specific covariates like $\text{volume}_t1$. The `alpha_v` and `beta_v` posterior samples encode which features drive variance: large posterior means for some `beta_v` components suggest those covariates strongly modulate noise. If `beta_v` is close to zero (posterior heavily peaked near 0), the data may not support heteroscedastic structure and the simpler constant-variance models are favored.

### Model 5. Bayesian Neural Network via mean-field SVI
Forest growth is likely affected by nonlinear interactions between variables such as initial height, biomass, species composition, moisture and local site conditions.

To test whether a nonlinear Bayesian model improves prediction, we implement a small Bayesian neural network (BNN). The model uses one hidden layer with a `tanh` activation function,

$$
f_\theta(x_i) = W_2 \tanh(W_1 x_i + b_1) + b_2,\
$$

where \(x_i\) is the feature vector for pixel \(i\). In contrast to a standard neural network, the weights and biases are not treated as fixed parameters. Instead, we place prior distributions over them:

$$W_1, b_1, W_2, b_2 \sim \mathcal{N}(0, 1).$$

The observed growth target is then modelled as

$$
y_i \sim \mathcal{N}(f_\theta(x_i), \sigma),
$$

with

$$
\sigma \sim \text{HalfNormal}(1).
$$

This means the BNN produces a posterior distribution over both the network parameters and the predicted growth. We fit the model using stochastic variational inference (SVI) with an `AutoDiagonalNormal` guide, which approximates the posterior with a mean-field Gaussian distribution.

The purpose of this model is not mainly interpretability, since individual neural network weights are difficult to interpret directly. Instead, the BNN is used as a Bayesian nonlinear benchmark: it tests whether allowing nonlinear relationships improves predictive performance compared with the Bayesian linear and heteroscedastic models, while still providing posterior predictive uncertainty.

In [ ]:
import torch.nn as nn

from pyro.nn import PyroModule, PyroSample
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


class BNN(PyroModule):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()

        self.hidden = PyroModule[nn.Linear](input_dim, hidden_dim)
        self.hidden.weight = PyroSample(dist.Normal(0.0, 1.0).expand([hidden_dim, input_dim]).to_event(2))
        self.hidden.bias = PyroSample(dist.Normal(0.0, 1.0).expand([hidden_dim]).to_event(1))

        self.out = PyroModule[nn.Linear](hidden_dim, 1)
        self.out.weight = PyroSample(dist.Normal(0.0, 1.0).expand([1, hidden_dim]).to_event(2))
        self.out.bias = PyroSample(dist.Normal(0.0, 1.0).expand([1]).to_event(1))

    def forward(self, X, y=None):
        hidden = torch.tanh(self.hidden(X))
        mean = self.out(hidden).squeeze(-1)

        pyro.deterministic("mean", mean)

        sigma = pyro.sample("sigma", dist.HalfNormal(1.0))

        with pyro.plate("data", X.shape[0]):
            pyro.sample("y", dist.Normal(mean, sigma), obs=y)

        return mean

In [31]:
BNN_N_STEPS = 2000
BNN_LR = 0.005
BNN_N_POSTERIOR_SAMPLES = 300
BNN_HIDDEN_DIM = 16

rows_bnn = []

for n_cells in SCALING_GRID:
    print(f"\nTraining BNN with n_cells = {n_cells}")

    # Get the same train/test split used by the other models
    s = slice_step(n_cells)

    X_train = torch.tensor(s["X_train"], dtype=torch.float32)
    y_train = torch.tensor(s["y_train"], dtype=torch.float32)
    X_test = torch.tensor(s["X_test"], dtype=torch.float32)

    # Reset Pyro before each model fit
    pyro.clear_param_store()
    pyro.set_rng_seed(SEED)
    torch.manual_seed(SEED)

    # Build model and guide
    model = BNN(
        input_dim=X_train.shape[1],
        hidden_dim=BNN_HIDDEN_DIM
    )

    guide = AutoDiagonalNormal(model)

    optimizer = Adam({"lr": BNN_LR})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    losses = []

    # Train with SVI
    for step in range(BNN_N_STEPS):
        loss = svi.step(X_train, y_train)
        losses.append(loss)

        if step % 250 == 0:
            print(f"  step {step:4d} | loss = {loss:.2f}")

    # Posterior predictive sampling
    predictive = Predictive(
        model,
        guide=guide,
        num_samples=BNN_N_POSTERIOR_SAMPLES,
        return_sites=("mean", "y", "sigma")
    )

    with torch.no_grad():
        pred = predictive(X_test)

    # Posterior mean prediction in standardised units
    y_hat_std = pred["mean"].mean(axis=0).numpy()

    # 90% posterior predictive interval in standardised units
    y_lower_std = torch.quantile(pred["y"], 0.05, dim=0).numpy()
    y_upper_std = torch.quantile(pred["y"], 0.95, dim=0).numpy()

    # Convert predictions and targets back to original units
    y_hat = y_hat_std * s["y_std"] + s["y_mean"]
    y_true = s["y_test"] * s["y_std"] + s["y_mean"]

    y_lower = y_lower_std * s["y_std"] + s["y_mean"]
    y_upper = y_upper_std * s["y_std"] + s["y_mean"]

    # Evaluate point predictions
    metrics = evaluate(y_true, y_hat, coords=s["coords_test"])

    # Prediction interval coverage
    pi90_coverage = np.mean((y_true >= y_lower) & (y_true <= y_upper))

    row = {
        "model": "BNN",
        "n_cells": n_cells,
        "n_train": s["n_train"],
        "n_test": s["n_test"],
        "final_loss": losses[-1],
        "sigma_mean_std_units": pred["sigma"].mean().item(),
        "PI90_coverage": pi90_coverage,
        **metrics,
    }

    rows_bnn.append(row)

df_bnn = pd.DataFrame(rows_bnn)
df_bnn


Training Simple BNN with n_cells = 5
  step    0 | loss = 5656.78
  step  250 | loss = 4633.72
  step  500 | loss = 4638.54
  step  750 | loss = 4451.84
  step 1000 | loss = 4554.44
  step 1250 | loss = 4284.38
  step 1500 | loss = 4186.03
  step 1750 | loss = 4098.99

Training Simple BNN with n_cells = 25
  step    0 | loss = 23687.73
  step  250 | loss = 19231.87
  step  500 | loss = 19310.48
  step  750 | loss = 18547.73
  step 1000 | loss = 18870.28
  step 1250 | loss = 17971.95
  step 1500 | loss = 17697.01
  step 1750 | loss = 17793.15

Training Simple BNN with n_cells = 50
  step    0 | loss = 44599.52
  step  250 | loss = 36353.57
  step  500 | loss = 36618.27
  step  750 | loss = 35216.26
  step 1000 | loss = 35690.20
  step 1250 | loss = 34100.09
  step 1500 | loss = 33592.76
  step 1750 | loss = 33865.65

Training Simple BNN with n_cells = all
  step    0 | loss = 79907.35
  step  250 | loss = 64735.22
  step  500 | loss = 65117.23
  step  750 | loss = 62491.26
  step 1000 

,model,n_cells,n_train,n_test,final_loss,sigma_mean_std_units,PI90_coverage,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,BNN,5,3411,10889,4071.166780,0.725991,0.902379,33.184880,23.711882,1.146502,0.403475,0.635767,0.168817,0.001
1,BNN,25,14945,10889,17577.106137,0.758312,0.921389,32.355305,23.205537,2.462648,0.432927,0.660647,0.154247,0.001
2,BNN,50,28925,10889,33379.990939,0.756539,0.924970,32.337806,23.218296,2.947530,0.433540,0.662142,0.154250,0.001
3,BNN,all,52184,10889,59395.322695,0.751162,0.924419,31.987457,22.792455,1.759546,0.445748,0.668934,0.143230,0.001


## 8. Model Comparison

**Overview.** This section assembles results from all five models — Ridge baseline, Exact conjugate inference, SVI, MCMC, and heteroscedastic regression — into a unified comparison table indexed by scaling step and model type. The goal is to answer key questions:

1. **Accuracy and scalability:** How does RMSE, MAE, and R² change across models and scaling steps? Does more data improve all models uniformly, or do some hit a ceiling?
2. **Inference-method agreement:** Do Exact, SVI, and MCMC (all fitting the *same* model) produce similar predictions? If not, which model's assumptions are problematic?
3. **Heteroscedasticity:** Does allowing variance to depend on $\mathbf{X}$ improve calibration (lower RMSE) or is it overfitting?
4. **Spatial diagnostics:** Does Moran's I stay high and stable across models, confirming that no linear model (even heteroscedastic) captures spatial structure?

The comparison respects the nested structure of the scaling grid: direct RMSE comparisons are valid only across models for the *same* `n_cells` value, since test set and training set size are identical at each step.


In [38]:
# Build a single comparison table for all models so the diagnostics below can reuse it.
# This cell should run before the comparison summaries.

df_ridge = pd.DataFrame(rows)
df_ridge["model"] = "Ridge"

df_exact = pd.DataFrame(rows)
df_exact["model"] = "Exact"

df_svi = pd.DataFrame(rows_svi)
df_svi["model"] = "SVI"

df_mcmc = pd.DataFrame(rows_mcmc)
df_mcmc["model"] = "MCMC"

df_hetero = pd.DataFrame(rows_hetero)
df_hetero["model"] = "Heteroscedastic"

df_bnn = pd.DataFrame(rows_bnn)
df_bnn["model"] = "BNN"

common_cols = ["n_cells", "n_train", "n_test", "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]
df_all_models = pd.concat(
    [
        df_ridge[common_cols + ["model"]],
        df_exact[common_cols + ["model"]],
        df_svi[common_cols + ["model"]],
        df_mcmc[common_cols + ["model"]],
        df_hetero[common_cols + ["model"]],
        df_bnn[common_cols + ["model"]],
    ],
    ignore_index=True,
    sort=False,
)

df_all_models = df_all_models[["model", "n_cells", "n_train", "n_test", "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]]
print("Comparison table ready:")
print(df_all_models.shape)


Comparison table ready:
(21, 11)


In [39]:
# Spatial diagnostics: Moran's I across models and scaling steps
print("\n" + "=" * 120)
print("SPATIAL DIAGNOSTICS: Moran's I Residual Autocorrelation")
print("=" * 120)

spatial_summary = df_all_models[["model", "n_cells", "MoranI", "MoranP"]].copy()
spatial_summary = spatial_summary.sort_values(["n_cells", "MoranI"], ascending=[True, False])
print(spatial_summary.to_string(index=False))
print("\nInterpretation:")
print("  • MoranI ≈ 0.18–0.20 with p = 0.001 across all models:")
print("    Linear models (even heteroscedastic) do not capture spatial autocorrelation.")
print("    Residuals are spatially correlated ⟹ hierarchical, spatial-lag, SAR, or GP models should absorb this signal.")
print("  • If MoranI drops significantly in later models (hierarchical/spatial), spatial structure was real and important.")


SPATIAL DIAGNOSTICS: Moran's I Residual Autocorrelation
          model n_cells   MoranI  MoranP
Heteroscedastic       5 0.214754   0.001
            SVI       5 0.195506   0.001
           MCMC       5 0.179990   0.001
          Ridge       5 0.178394   0.001
          Exact       5 0.178394   0.001
            BNN       5 0.168817   0.001
Heteroscedastic      25 0.220913   0.001
            SVI      25 0.195427   0.001
          Ridge      25 0.180892   0.001
          Exact      25 0.180892   0.001
            BNN      25 0.154247   0.001
Heteroscedastic      50 0.212179   0.001
            SVI      50 0.195361   0.001
          Ridge      50 0.180873   0.001
          Exact      50 0.180873   0.001
            BNN      50 0.154250   0.001
Heteroscedastic     all 0.209355   0.001
            SVI     all 0.193364   0.001
          Ridge     all 0.178661   0.001
          Exact     all 0.178661   0.001
            BNN     all 0.143230   0.001

Interpretation:
  • MoranI ≈ 0.18–0.20 w

In [34]:
# Heteroscedasticity check: does allowing variance to depend on X help?
print("\n" + "=" * 120)
print("HETEROSCEDASTICITY CHECK: Does Heteroscedastic regression improve over SVI?")
print("=" * 120)

if 'df_all_models' not in globals():
    # Build the comparison table on demand so this cell works even if
    # executed before the full comparison-summary cell below.
    df_ridge = pd.DataFrame(rows)
    df_ridge["model"] = "Ridge"
    df_exact = pd.DataFrame(rows)
    df_exact["model"] = "Exact"
    df_svi = pd.DataFrame(rows_svi)
    df_svi["model"] = "SVI"
    df_mcmc = pd.DataFrame(rows_mcmc)
    df_mcmc["model"] = "MCMC"
    df_hetero = pd.DataFrame(rows_hetero)
    df_hetero["model"] = "Heteroscedastic"
    common_cols = ["n_cells", "n_train", "n_test", "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]
    comparison_parts = [
        df_ridge[common_cols + ["model"]],
        df_exact[common_cols + ["model"]],
        df_svi[common_cols + ["model"]],
        df_mcmc[common_cols + ["model"]],
        df_hetero[common_cols + ["model"]],
    ]
    df_all_models = pd.concat(comparison_parts, ignore_index=True, sort=False)

hetero_check = []
for n_cells_val in SCALING_GRID:
    svi_row = df_all_models[(df_all_models["n_cells"] == n_cells_val) & (df_all_models["model"] == "SVI")]
    hetero_row = df_all_models[(df_all_models["n_cells"] == n_cells_val) & (df_all_models["model"] == "Heteroscedastic")]
    
    if len(svi_row) > 0 and len(hetero_row) > 0:
        svi_rmse = svi_row.iloc[0]["RMSE"]
        hetero_rmse = hetero_row.iloc[0]["RMSE"]
        svi_r2 = svi_row.iloc[0]["R2"]
        hetero_r2 = hetero_row.iloc[0]["R2"]
        
        rmse_improvement = (svi_rmse - hetero_rmse) / svi_rmse * 100  # % improvement
        r2_improvement = (hetero_r2 - svi_r2) * 100  # % point improvement
        
        hetero_check.append({
            "n_cells": n_cells_val,
            "SVI_RMSE": svi_rmse,
            "Hetero_RMSE": hetero_rmse,
            "RMSE_Impr_%": rmse_improvement,
            "SVI_R2": svi_r2,
            "Hetero_R2": hetero_r2,
            "R2_Impr_pp": r2_improvement,
        })

if hetero_check:
    df_hetero_check = pd.DataFrame(hetero_check)
    print(df_hetero_check.to_string(index=False))
    print("\nInterpretation:")
    print("  • Positive RMSE_Impr_%: Heteroscedastic model improves prediction error (overfitting unlikely).")
    print("  • Negative RMSE_Impr_%: Heteroscedastic model regresses (variance parameterization not justified by data).")
    print("  • Small improvement: Linear constant-variance model is adequate; additional variance parameters not worth complexity.")
else:
    print("(Heteroscedastic model not run)")


HETEROSCEDASTICITY CHECK: Does Heteroscedastic regression improve over SVI?
n_cells  SVI_RMSE  Hetero_RMSE  RMSE_Impr_%   SVI_R2  Hetero_R2  R2_Impr_pp
      5 34.411595    36.351223    -5.636553 0.358557   0.284209   -7.434841
     25 34.325040    36.452982    -6.199385 0.361780   0.280196   -8.158424
     50 34.220356    35.341236    -3.275476 0.365667   0.323432   -4.223540
    all 34.229703    35.330005    -3.214463 0.365321   0.323862   -4.145887

Interpretation:
  • Positive RMSE_Impr_%: Heteroscedastic model improves prediction error (overfitting unlikely).
  • Negative RMSE_Impr_%: Heteroscedastic model regresses (variance parameterization not justified by data).
  • Small improvement: Linear constant-variance model is adequate; additional variance parameters not worth complexity.


In [35]:
# Inference-method agreement (if MCMC was run on the full grid, compare SVI vs MCMC)
print("\n" + "=" * 120)
print("INFERENCE-METHOD AGREEMENT: SVI vs MCMC (same model, different inference)")
print("=" * 120)

svi_mcmc_compare = []
for n_cells_val in SCALING_GRID:
    svi_row = df_all_models[(df_all_models["n_cells"] == n_cells_val) & (df_all_models["model"] == "SVI")]
    mcmc_row = df_all_models[(df_all_models["n_cells"] == n_cells_val) & (df_all_models["model"] == "MCMC")]
    
    if len(svi_row) > 0 and len(mcmc_row) > 0:
        svi_rmse = svi_row.iloc[0]["RMSE"]
        mcmc_rmse = mcmc_row.iloc[0]["RMSE"]
        svi_r2 = svi_row.iloc[0]["R2"]
        mcmc_r2 = mcmc_row.iloc[0]["R2"]
        
        rmse_diff = abs(svi_rmse - mcmc_rmse)
        r2_diff = abs(svi_r2 - mcmc_r2)
        
        svi_mcmc_compare.append({
            "n_cells": n_cells_val,
            "SVI_RMSE": svi_rmse,
            "MCMC_RMSE": mcmc_rmse,
            "RMSE_Δ": rmse_diff,
            "SVI_R2": svi_r2,
            "MCMC_R2": mcmc_r2,
            "R2_Δ": r2_diff,
        })

if svi_mcmc_compare:
    df_agreement = pd.DataFrame(svi_mcmc_compare)
    print(df_agreement.to_string(index=False))
    print("\nInterpretation:")
    print("  • Small RMSE_Δ and R2_Δ: SVI's mean-field approximation is tight; VI is a good fast proxy for MCMC.")
    print("  • Large RMSE_Δ or R2_Δ: VI's independence assumption is problematic; posterior correlation structure matters.")
else:
    print("(MCMC not run on full grid — set RUN_MCMC_FULL_GRID=True to compute)")


INFERENCE-METHOD AGREEMENT: SVI vs MCMC (same model, different inference)
 n_cells  SVI_RMSE  MCMC_RMSE  RMSE_Δ   SVI_R2  MCMC_R2     R2_Δ
       5 34.411595  33.924225 0.48737 0.358557 0.376598 0.018041

Interpretation:
  • Small RMSE_Δ and R2_Δ: SVI's mean-field approximation is tight; VI is a good fast proxy for MCMC.
  • Large RMSE_Δ or R2_Δ: VI's independence assumption is problematic; posterior correlation structure matters.


In [36]:
# Per-scaling-step summary: which model is best on each metric?
print("\n" + "=" * 120)
print("PER SCALING STEP: BEST MODEL ON RMSE")
print("=" * 120)

for n_cells_val in SCALING_GRID:
    subset = df_all_models[df_all_models["n_cells"] == n_cells_val].copy()
    if len(subset) == 0:
        continue
    
    # Sort by RMSE, show top 3
    best_rmse = subset.nsmallest(3, "RMSE")[["model", "RMSE", "MAE", "R2", "Corr"]]
    print(f"\nn_cells = {n_cells_val}  |  n_train = {subset.iloc[0]['n_train']:.0f}  |  n_test = {subset.iloc[0]['n_test']:.0f}")
    print(best_rmse.to_string(index=False))


PER SCALING STEP: BEST MODEL ON RMSE

n_cells = 5  |  n_train = 3411  |  n_test = 10889
model      RMSE       MAE       R2     Corr
Ridge 33.921905 24.516541 0.376683 0.614876
Exact 33.921905 24.516541 0.376683 0.614876
 MCMC 33.924225 24.522583 0.376598 0.614653

n_cells = 25  |  n_train = 14945  |  n_test = 10889
model      RMSE       MAE       R2     Corr
Ridge 34.073374 24.590349 0.371105 0.611753
Exact 34.073374 24.590349 0.371105 0.611753
  SVI 34.325040 24.664921 0.361780 0.602069

n_cells = 50  |  n_train = 28925  |  n_test = 10889
model      RMSE       MAE       R2     Corr
Ridge 33.731277 24.508398 0.383669 0.621540
Exact 33.731277 24.508398 0.383669 0.621540
  SVI 34.220356 24.622264 0.365667 0.605626

n_cells = all  |  n_train = 52184  |  n_test = 10889
model      RMSE       MAE       R2     Corr
Ridge 33.716478 24.518734 0.384210 0.621744
Exact 33.716478 24.518734 0.384210 0.621744
  SVI 34.229703 24.710449 0.365321 0.605697


In [37]:
# Consolidate results from all models into separate comparison tables per scaling step
# Each row represents one model + one scaling step

# Ridge baseline (add model name)
df_ridge = pd.DataFrame(rows)
df_ridge["model"] = "Ridge"

# Exact conjugate inference (rows already have the structure, but we need to reconstruct)
# We need to re-run the exact inference to capture results, or extract from the existing run
# Since exact_bayesian_linear_regression was run and stored in rows, let's use it directly
df_exact = pd.DataFrame(rows)  # This contains exact results
df_exact["model"] = "Exact"

# SVI
df_svi = pd.DataFrame(rows_svi)
df_svi["model"] = "SVI"

# MCMC
df_mcmc = pd.DataFrame(rows_mcmc)
df_mcmc["model"] = "MCMC"

# Heteroscedastic
df_hetero = pd.DataFrame(rows_hetero)
df_hetero["model"] = "Heteroscedastic"

# Combine all into one comparison dataframe
# Select common columns for comparison
common_cols = ["n_cells", "n_train", "n_test", "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]
comparison_parts = []

if "RMSE" in df_ridge.columns:
    comparison_parts.append(df_ridge[common_cols + ["model"]])
if "RMSE" in df_exact.columns:
    comparison_parts.append(df_exact[common_cols + ["model"]])
if "RMSE" in df_svi.columns:
    comparison_parts.append(df_svi[common_cols + ["model"]])
if "RMSE" in df_mcmc.columns:
    comparison_parts.append(df_mcmc[common_cols + ["model"]])
if "RMSE" in df_hetero.columns:
    comparison_parts.append(df_hetero[common_cols + ["model"]])

df_all_models = pd.concat(comparison_parts, ignore_index=True, sort=False)

# Reorder columns for readability
df_all_models = df_all_models[["model", "n_cells", "n_train", "n_test", "RMSE", "MAE", "Bias", "R2", "Corr", "MoranI", "MoranP"]]

print("=" * 120)
print("FULL MODEL COMPARISON ACROSS ALL SCALING STEPS")
print("=" * 120)
df_all_models

FULL MODEL COMPARISON ACROSS ALL SCALING STEPS


,model,n_cells,n_train,n_test,RMSE,MAE,Bias,R2,Corr,MoranI,MoranP
0,Ridge,5,3411,10889,33.921905,24.516541,-0.495750,0.376683,0.614876,0.178394,0.001
1,Ridge,25,14945,10889,34.073374,24.590349,1.678498,0.371105,0.611753,0.180892,0.001
2,Ridge,50,28925,10889,33.731277,24.508398,1.834485,0.383669,0.621540,0.180873,0.001
3,Ridge,all,52184,10889,33.716478,24.518734,1.975397,0.384210,0.621744,0.178661,0.001
4,Exact,5,3411,10889,33.921905,24.516541,-0.495750,0.376683,0.614876,0.178394,0.001
5,Exact,25,14945,10889,34.073374,24.590349,1.678498,0.371105,0.611753,0.180892,0.001
6,Exact,50,28925,10889,33.731277,24.508398,1.834485,0.383669,0.621540,0.180873,0.001
7,Exact,all,52184,10889,33.716478,24.518734,1.975397,0.384210,0.621744,0.178661,0.001
8,SVI,5,3411,10889,34.411595,24.646367,-0.636305,0.358557,0.599056,0.195506,0.001
9,SVI,25,14945,10889,34.325040,24.664921,1.140254,0.361780,0.602069,0.195427,0.001
